# EVO test

In [ ]:
import pandas as pd
import os

## Load data

In [2]:
base_path = "../../data/perphect-data"

couples_df = pd.read_csv(os.path.join(base_path, "public_data_set", "couples_df.csv"))
phages_df = pd.read_csv(os.path.join(base_path, "public_data_set", "phages_df.csv"))
bacteria_df = pd.read_csv(os.path.join(base_path, "public_data_set", "bacteria_df.csv"))
print('couples_df.shape = ', couples_df.shape)
print('phages_df.shape = ', phages_df.shape)
print('bacteria_df.shape = ', bacteria_df.shape)

couples_df.shape =  (4202, 4)
phages_df.shape =  (3459, 3)
bacteria_df.shape =  (94, 3)


In [3]:
phages_df.head()

,phage_id,phage_sequence,sequence_length
0,5326,TGCGGCCGCCCCATCCTGTACGGGTTTCCAAGTCGATCGGAGGGCA...,53124
1,6247,GGCTTTCGTGTGAGCCGTGATGTTTTCACGAATATGTGCCCCACCT...,74483
2,1976,GTGGGAATTTTTTTTTTGGGTTGCGCGGTGATCGCCGATGACGACG...,50781
3,430,ATGGCTTCGACTCAGACTCCAGCCGTCGGCAAGACCACGGCCATCG...,71565
4,431,TGCGGCTGAGCCATCGTGTACGGGTTTCCAAGTCCATCAGAGCCGG...,53396


## Embeddings

In [1]:
from evo import Evo
import torch

/home/carrillo/micromamba/envs/pbi-evo/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# device = 'cuda:0'
device = 'cpu'

evo_model = Evo('evo-1-8k-base', device=device)
model, tokenizer = evo_model.model, evo_model.tokenizer

Loading checkpoint shards: 100%|██████████| 3/3 [00:14<00:00,  4.67s/it]


In [ ]:
model.to(device)
model.eval()

In [ ]:
# monkey patch the unembed function with identity
# this removes the final projection back from the embedding space into tokens
# so the "logits" of the model is now the final layer embedding
# see source for unembed - https://huggingface.co/togethercomputer/evo-1-131k-base/blob/main/model.py#L339

from torch import nn

class CustomEmbedding(nn.Module):
  def unembed(self, u):
    return u

model.unembed = CustomEmbedding()

# end custom code

In [ ]:
sequence = 'ACGT'
input_ids = torch.tensor(
    tokenizer.tokenize(sequence),
    dtype=torch.int,
).to(device).unsqueeze(0)

In [ ]:
embed, _ = model(input_ids) # (batch, length, embed dim)

print('Embed: ', embed)
print('Shape (batch, length, embed dim): ', embed.shape)

In [ ]:
# you can now use embedding for downstream classification tasks
# you probably want to aggregate over position dimension
mean_emb = embed.mean(dim=1)

Embeddings shape: (2, 2048, 512)
Mean sequence embeddings: tensor([[ 0.0483,  0.2019,  0.2050,  ..., -0.1056,  0.5363, -0.4475],
        [-0.3753,  0.2040,  0.0233,  ...,  0.0192,  0.1604, -0.2348]])
